# VideoSender And VideoSenderOptions

`pyneat.groups.video_sender(...)` is the customer-facing **video egress** fragment of a Neat graph.
It takes video out of your application and puts it on the network as **H.264 over RTP/UDP**, which is
exactly what the Insight viewer, and most recorders and analytics sinks, expect.

This notebook answers four practical questions:

- What does `video_sender` actually build behind the scenes?
- What is the difference between the **raw** and **encoded** input kinds?
- What does every field in `VideoSenderOptions`, `VideoSenderEncoderOptions`, and `VideoSenderRtpOptions` do?
- How do you push frames into it and confirm the bytes are really leaving the board?

Model configuration belongs to `ModelOptions` (notebook 01), stream ingest belongs to
`RtspDecodedInputOptions` (notebook 02), and JSON metadata egress belongs to `MetadataSender` (notebook 06).
This notebook is only about video egress.

## Mental Model

`video_sender` is the mirror image of `rtsp_decoded_input`. One brings pixels in, the other pushes pixels out.

```text
rtsp_decoded_input :   network H.264  ->  decode  ->  raw NV12 frames
video_sender       :   raw NV12 frames ->  encode  ->  network H.264
```

The group is not a single node. It is a small graph fragment that Neat assembles for you.
With **raw** input it builds seven stages:

```text
your NV12 frame
  -> caps (width, height, fps, any memory)
  -> videoconvert
  -> caps (NV12, width, height, fps)
  -> SiMa hardware H.264 encoder   (bitrate_kbps, profile, level)
  -> H.264 parser                  (config_interval)
  -> RTP H.264 packetizer          (payload_type, config_interval)
  -> UDP output                    (host, video_port_base + channel)
```

With **encoded** input it skips the first four stages and only builds the last three:

```text
your already-H.264 bytes
  -> H.264 parser -> RTP H.264 packetizer -> UDP output
```

Everything you configure on `VideoSenderOptions` lands on one of those stages.

## Imports

Run this notebook from the DevKit pyneat environment.

In [1]:
import json
import time
import socket
import urllib.request

import cv2
import numpy as np
import pyneat

## Notebook Settings

`video_sender` sends UDP to a host and port. For the Insight viewer, that host is the machine
running Insight (the Neat Development Environment host), and the port follows a fixed channel rule.

| Setting | Meaning |
| --- | --- |
| `INSIGHT_HOST` | IP of the machine running Insight. This must be reachable from the DevKit. |
| `CHANNEL` | Insight viewer channel, `0`-`79`. Selects the port and the viewer tile. |
| `VIDEO_PORT_BASE` | Base UDP port for video. Insight's default is `9000`. |
| `FRAME_WIDTH/HEIGHT/FPS` | The contract you promise the encoder. Every frame you push must match it. |

The port rule is `video_port = video_port_base + channel`. Nothing else changes it.

In [2]:
INSIGHT_HOST = "192.168.131.12"  # Machine running Insight. Replace with your host IP.
CHANNEL = 0  # Insight viewer channel 0-79.
VIDEO_PORT_BASE = 9000  # Insight video UDP base port.

FRAME_WIDTH = 1280
FRAME_HEIGHT = 720
FRAME_FPS = 30

INSIGHT_API = f"https://{INSIGHT_HOST}:9900"  # Used only for the optional verification cells.

print("video will be sent to:", f"{INSIGHT_HOST}:{VIDEO_PORT_BASE + CHANNEL}")

video will be sent to: 192.168.131.12:9000


## Public Fields

As in notebook 01, start from discovery rather than a hand-written list. Three option objects make up
the video sender surface, and two of them are nested inside the third.

In [3]:
def public_names(obj):
    out = [name for name in dir(obj) if not name.startswith("_")]
    return out, len(out)


probe = pyneat.VideoSenderOptions.h264_rtp_udp_from_raw(FRAME_WIDTH, FRAME_HEIGHT, FRAME_FPS)

print("VideoSenderOptions:")
print(public_names(probe))
print()
print("VideoSenderEncoderOptions (probe.encoder):")
print(public_names(probe.encoder))
print()
print("VideoSenderRtpOptions (probe.rtp):")
print(public_names(probe.rtp))

VideoSenderOptions:
(['async', 'async_', 'channel', 'encoder', 'fps', 'h264_rtp_udp_from_encoded', 'h264_rtp_udp_from_raw', 'height', 'host', 'is_encoded_input', 'is_raw_input', 'rtp', 'sync', 'video_port', 'video_port_base', 'width'], 16)

VideoSenderEncoderOptions (probe.encoder):
(['bitrate_kbps', 'level', 'profile'], 3)

VideoSenderRtpOptions (probe.rtp):
(['config_interval', 'payload_type'], 2)


## Complete VideoSenderOptions Field Map

### Root Fields

| Field | Type / default | One-line use |
| --- | --- | --- |
| `host` | `str`, `"127.0.0.1"` | Destination IP for the RTP/UDP stream. |
| `channel` | `int`, `0` | Logical stream index. Added to `video_port_base` to get the real port. |
| `video_port_base` | `int`, `9000` | Base UDP port. Insight listens on `9000-9079`. |
| `sync` | `bool`, `False` | Pace the UDP sink to the pipeline clock. Leave `False` for live analytics. |
| `async_` | `bool`, `False` | Let the UDP sink pre-roll asynchronously. Leave `False` for live analytics. |
| `rtp` | `VideoSenderRtpOptions` | RTP packetization settings. |
| `encoder` | `VideoSenderEncoderOptions` | H.264 encoder settings. Ignored for encoded input. |

### Read-Only Properties

These are set by the factory constructor and cannot be assigned afterwards.

| Property | Meaning |
| --- | --- |
| `width`, `height`, `fps` | The raw-frame contract. `0` for the encoded-input kind. |
| `video_port` | Computed `video_port_base + channel`. Always read the port from here. |
| `is_raw_input()` | `True` when the group will encode for you. |
| `is_encoded_input()` | `True` when the group expects H.264 bytes already. |

### `encoder` - `VideoSenderEncoderOptions`

| Field | Default | Use |
| --- | --- | --- |
| `bitrate_kbps` | `4000` | Target H.264 bitrate. Raise for detail, lower for constrained links. |
| `profile` | `"baseline"` | H.264 profile. `baseline` is the most compatible with browser/WebRTC receivers. |
| `level` | `"4.0"` | H.264 level. Must be high enough for your resolution and frame rate. |

### `rtp` - `VideoSenderRtpOptions`

| Field | Default | Use |
| --- | --- | --- |
| `payload_type` | `96` | RTP dynamic payload type for H.264. Receivers expect `96` unless told otherwise. |
| `config_interval` | `1` | Seconds between SPS/PPS reinjection. **This is the field that lets a late viewer start decoding.** |

`config_interval` deserves attention. SPS/PPS are the stream headers a decoder needs before it can
decode anything. If they are sent once at startup, a viewer that opens the page thirty seconds later
sees nothing but silence. `config_interval = 1` re-sends them every second, so any viewer can join at
any time and start decoding within a second. Setting it to `-1` disables reinjection; only do that for
a receiver you control that is guaranteed to be listening before the first frame.

## Two Input Kinds

`VideoSenderOptions` has no public constructor. You pick the input kind with a factory, and the factory
decides what the group builds.

| Factory | Input you push | Group encodes? | Typical source |
| --- | --- | --- | --- |
| `VideoSenderOptions.h264_rtp_udp_from_raw(width, height, fps)` | Raw frames, normally NV12 | Yes, on the SiMa hardware encoder | Decoded and/or annotated frames from your app. |
| `VideoSenderOptions.h264_rtp_udp_from_encoded()` | H.264 bytes | No | An H.264 elementary stream you already have, or a pass-through from `rtsp_encoded_input`. |

Use **raw** whenever your application touched the pixels. Use **encoded** only for pure remuxing, where
you want to move existing H.264 onto RTP/UDP without paying for a decode/encode round trip.

In [4]:
raw_opt = pyneat.VideoSenderOptions.h264_rtp_udp_from_raw(FRAME_WIDTH, FRAME_HEIGHT, FRAME_FPS)
encoded_opt = pyneat.VideoSenderOptions.h264_rtp_udp_from_encoded()

print("raw input kind")
print("  is_raw_input :", raw_opt.is_raw_input())
print("  is_encoded_input:", raw_opt.is_encoded_input())
print("  width/height/fps:", raw_opt.width, raw_opt.height, raw_opt.fps)
print()
print("encoded input kind")
print("  is_raw_input :", encoded_opt.is_raw_input())
print("  is_encoded_input:", encoded_opt.is_encoded_input())
print("  width/height/fps:", encoded_opt.width, encoded_opt.height, encoded_opt.fps)

raw input kind
  is_raw_input : True
  is_encoded_input: False
  width/height/fps: 1280 720 30

encoded input kind
  is_raw_input : False
  is_encoded_input: True
  width/height/fps: 0 0 0


The factory validates its arguments. `width`, `height`, and `fps` must all be positive, because the
encoder and the caps filters ahead of it need a concrete contract.

In [5]:
for bad in [(0, 720, 30), (1280, 0, 30), (1280, 720, 0)]:
    try:
        pyneat.VideoSenderOptions.h264_rtp_udp_from_raw(*bad)
        print(bad, "-> accepted (unexpected)")
    except Exception as exc:
        print(bad, "->", type(exc).__name__, exc)

(0, 720, 30) -> ValueError VideoSenderOptions: width must be > 0
(1280, 0, 30) -> ValueError VideoSenderOptions: height must be > 0
(1280, 720, 0) -> ValueError VideoSenderOptions: fps must be > 0


## Channel And Port Math

The channel is the only thing you should change when you add a stream. Never hand-compute the port and
never assign to `video_port`: it is read-only and derived.

For Insight, channel `N` means video on `9000 + N` **and** metadata on `9100 + N`. Keeping those two
numbers equal is what makes the viewer draw your boxes on top of the right video tile. Notebook 06
covers the metadata half.

In [6]:
def make_sender_options(channel: int, width: int, height: int, fps: int):
    opt = pyneat.VideoSenderOptions.h264_rtp_udp_from_raw(width, height, fps)
    opt.host = INSIGHT_HOST  # Where Insight is listening.
    opt.channel = channel  # Viewer channel/tile.
    opt.video_port_base = VIDEO_PORT_BASE  # Insight video base port.
    opt.sync = False  # Live analytics: do not pace to the clock.
    opt.async_ = False  # Live analytics: do not pre-roll.
    opt.encoder.bitrate_kbps = 4000  # Target bitrate.
    opt.encoder.profile = "baseline"  # Widest receiver compatibility.
    opt.encoder.level = "4.0"  # Enough for 1280x720.
    opt.rtp.payload_type = 96  # Standard dynamic payload type for H.264.
    opt.rtp.config_interval = 1  # Re-send SPS/PPS every second so late viewers can join.
    return opt


for ch in [0, 1, 2, 7]:
    opt = make_sender_options(ch, FRAME_WIDTH, FRAME_HEIGHT, FRAME_FPS)
    print(f"channel {ch}: video udp {opt.host}:{opt.video_port}  (metadata would be {9100 + ch})")

channel 0: video udp 192.168.131.12:9000  (metadata would be 9100)
channel 1: video udp 192.168.131.12:9001  (metadata would be 9101)
channel 2: video udp 192.168.131.12:9002  (metadata would be 9102)
channel 7: video udp 192.168.131.12:9007  (metadata would be 9107)


## Encoder Options In Practice

The encoder fields are only consulted for the **raw** input kind. Reasonable starting points:

| Resolution / rate | `bitrate_kbps` | `level` | Note |
| --- | --- | --- | --- |
| 640x480 @ 30 | `1500` | `"3.1"` | Small tiles, multi-stream viewer grids. |
| 1280x720 @ 30 | `4000` | `"4.0"` | The default, and the right answer for most demos. |
| 1280x720 @ 60 | `6000` | `"4.2"` | Doubling the frame rate needs more bits for the same quality. |
| 1920x1080 @ 30 | `8000` | `"4.2"` | Detail-heavy scenes; watch the network. |

Two rules that save time:

- **Bitrate is a quality/bandwidth trade, not a correctness knob.** If the viewer shows blocky mush,
  raise it. If UDP is dropping packets, lower it before you blame the decoder.
- **Level must cover resolution x frame rate.** A level that is too low is a real encoder error, not a
  cosmetic one. When unsure, `"4.2"` is a safe ceiling for 1080p30 and below.

Prefer `baseline` profile for anything a browser will render. Insight decodes in the browser via WebRTC,
and `baseline` avoids B-frames and the latency they add.

In [7]:
low_bandwidth = pyneat.VideoSenderOptions.h264_rtp_udp_from_raw(640, 480, 30)
low_bandwidth.encoder.bitrate_kbps = 1500
low_bandwidth.encoder.level = "3.1"

high_detail = pyneat.VideoSenderOptions.h264_rtp_udp_from_raw(1920, 1080, 30)
high_detail.encoder.bitrate_kbps = 8000
high_detail.encoder.level = "4.2"

for name, opt in [("low_bandwidth", low_bandwidth), ("high_detail", high_detail)]:
    print(f"{name}: {opt.width}x{opt.height}@{opt.fps} "
          f"bitrate={opt.encoder.bitrate_kbps}kbps profile={opt.encoder.profile} level={opt.encoder.level}")

low_bandwidth: 640x480@30 bitrate=1500kbps profile=baseline level=3.1
high_detail: 1920x1080@30 bitrate=8000kbps profile=baseline level=4.2


## `sync` And `async_`

These two booleans reach the UDP sink underneath the group. They exist because GStreamer sinks can
either render on the pipeline clock or push as fast as data arrives.

| Field | `False` (default) | `True` |
| --- | --- | --- |
| `sync` | Send each frame as soon as it is encoded. | Hold each frame until its presentation time. |
| `async_` | Sink goes to `PLAYING` immediately. | Sink pre-rolls and waits for the first buffer. |

For live analytics, leave both `False`. Your frames are already paced by the camera, so a second layer
of clock pacing only adds latency and gives the pipeline a way to stall. Turn `sync` on only when you
are pushing frames from a file faster than real time and want the receiver to see real-time playback.

Note the trailing underscore: `async` is a Python keyword, so the binding exposes `async_`.
A plain `async` property is also provided as an alias for convenience.

In [8]:
opt = make_sender_options(CHANNEL, FRAME_WIDTH, FRAME_HEIGHT, FRAME_FPS)
print("sync :", opt.sync)
print("async_:", opt.async_)

opt.async_ = True
print("alias reads the same value:", opt.async_, getattr(opt, "async"))
opt.async_ = False

sync : False
async_: False
alias reads the same value: True True


## The Input Node: What You Must Promise The Encoder

`video_sender` is a sink fragment. It has no input node of its own, so a graph that only contains
`video_sender` has nothing to push into. You pair it with `pyneat.nodes.input(...)` and an
`InputOptions` object that declares exactly the frames you will push.

This is the single most common source of failure in a video egress pipeline. The `InputOptions` caps,
the `h264_rtp_udp_from_raw` arguments, and the tensors you actually push must all agree on format,
width, height, and frame rate. If they disagree, the pipeline either refuses to negotiate or produces
garbage.

In [9]:
def make_nv12_input_options(width: int, height: int, fps: int):
    input_opt = pyneat.InputOptions()
    input_opt.payload_type = pyneat.PayloadType.Image  # Pushing image frames, not model tensors.
    input_opt.format = pyneat.Format.NV12  # Encoder-native format; no conversion cost.
    input_opt.width = width
    input_opt.height = height
    input_opt.depth = 1  # NV12 is a single packed plane pair, not 3 separate channels.
    input_opt.max_width = width
    input_opt.max_height = height
    input_opt.max_depth = 1
    input_opt.fps_n = max(1, fps)  # Frame rate numerator.
    input_opt.fps_d = 1  # Frame rate denominator.
    input_opt.caps_override = (
        f"video/x-raw,format=NV12,width={width},height={height},framerate={max(1, fps)}/1"
    )  # Be explicit; leaves nothing for caps negotiation to guess wrong.
    input_opt.use_simaai_pool = False  # Frames come from CPU/numpy memory in this notebook.
    return input_opt


nv12_input = make_nv12_input_options(FRAME_WIDTH, FRAME_HEIGHT, FRAME_FPS)
print("caps:", nv12_input.caps_override)

caps: video/x-raw,format=NV12,width=1280,height=720,framerate=30/1


## NV12 Tensor Helpers

The encoder wants a real NV12 tensor, with `ImageSpec` and Y/UV `Plane` metadata attached, not a bare
two-dimensional numpy array. A numpy array of the right shape will be accepted as a buffer of bytes and
then interpreted incorrectly.

NV12 stores a full-resolution Y (luma) plane followed by a half-resolution interleaved UV (chroma)
plane, giving `height * 3 / 2` rows in total.

```text
row 0            +-----------------------------+
                 |                             |
                 |    Y plane (height x width) |
                 |                             |
row height       +-----------------------------+
                 |  UV plane (height/2 x width)|
row height*3/2   +-----------------------------+
```

These two helpers are the ones the demo applications use. Keep them; they are correct, and getting the
plane strides and byte offsets right by hand is fiddly.

In [10]:
def bgr_to_nv12(frame_bgr):
    height, width = frame_bgr.shape[:2]
    if height % 2 != 0 or width % 2 != 0:
        raise RuntimeError(f"NV12 requires even dimensions, got {width}x{height}")
    i420 = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2YUV_I420).reshape(-1)
    y_size = width * height
    uv_size = y_size // 4
    y = i420[:y_size].reshape(height, width)
    u = i420[y_size:y_size + uv_size].reshape(height // 2, width // 2)
    v = i420[y_size + uv_size:y_size + uv_size * 2].reshape(height // 2, width // 2)
    uv = np.empty((height // 2, width), dtype=np.uint8)
    uv[:, 0::2] = u  # NV12 interleaves U and V; I420 keeps them in separate planes.
    uv[:, 1::2] = v
    return np.ascontiguousarray(np.vstack((y, uv)))


def make_nv12_tensor(nv12, width: int, height: int):
    tensor = pyneat.Tensor.from_numpy(
        np.ascontiguousarray(nv12),
        copy=True,
        layout=pyneat.TensorLayout.HW,
        memory=pyneat.TensorMemory.CPU,
    )
    tensor.shape = [height, width]
    tensor.strides_bytes = [width, 1]
    tensor.byte_offset = 0

    image = pyneat.ImageSpec()
    image.format = pyneat.PixelFormat.NV12
    semantic = tensor.semantic
    semantic.image = image
    tensor.semantic = semantic  # Declares "these bytes are NV12 pixels".

    y = pyneat.Plane()
    y.role = pyneat.PlaneRole.Y
    y.shape = [height, width]
    y.strides_bytes = [width, 1]
    y.byte_offset = 0

    uv = pyneat.Plane()
    uv.role = pyneat.PlaneRole.UV
    uv.shape = [height // 2, width]
    uv.strides_bytes = [width, 1]
    uv.byte_offset = width * height  # UV starts right after the Y plane.

    tensor.planes = [y, uv]
    return tensor


demo_bgr = np.zeros((FRAME_HEIGHT, FRAME_WIDTH, 3), dtype=np.uint8)
demo_nv12 = bgr_to_nv12(demo_bgr)
demo_tensor = make_nv12_tensor(demo_nv12, FRAME_WIDTH, FRAME_HEIGHT)

print("nv12 numpy shape:", demo_nv12.shape, "expected:", (FRAME_HEIGHT * 3 // 2, FRAME_WIDTH))
print("tensor shape :", demo_tensor.shape)
print("tensor planes:", [(p.role, p.shape, p.byte_offset) for p in demo_tensor.planes])

nv12 numpy shape: (1080, 1280) expected: (1080, 1280)
tensor shape : [720, 1280]
tensor planes: [(PlaneRole.Y, [720, 1280], 0), (PlaneRole.UV, [360, 1280], 921600)]


## Build The Video Sender Graph

A minimal egress graph is two things: an input node that accepts your NV12 frames, and the
`video_sender` group that encodes and ships them.

```text
pyneat.nodes.input(nv12_input)  ->  pyneat.groups.video_sender(sender_opt)
```

`graph.build([seed_tensor])` builds the graph and negotiates caps using a seed frame. Seeding the build
with a real NV12 tensor of the right size means the encoder is fully negotiated before your first real
frame arrives, so the first push is not also the first negotiation.

In [11]:
sender_opt = make_sender_options(CHANNEL, FRAME_WIDTH, FRAME_HEIGHT, FRAME_FPS)

video_graph = pyneat.Graph("video_sender_demo")
video_graph.add(pyneat.nodes.input(make_nv12_input_options(FRAME_WIDTH, FRAME_HEIGHT, FRAME_FPS)))
video_graph.add(pyneat.groups.video_sender(sender_opt))

print(video_graph.describe())
print("destination:", f"{sender_opt.host}:{sender_opt.video_port}")

0) Input  [mysrc]
1) CapsRaw
2) VideoConvert
3) CapsRaw
4) H264EncodeSima
5) H264Parse
6) H264Packetize
7) UdpOutput
destination: 192.168.131.12:9000


`describe_backend()` shows the actual element chain the group expanded into. This is the fastest way to
confirm that the encoder, parser, packetizer, and UDP sink were all created with the settings you meant.

In [12]:
print(video_graph.describe_backend())

appsrc name=mysrc_1 is-live=true format=time do-timestamp=true block=true stream-type=stream max-bytes=0 caps="video/x-raw,format=NV12,width=1280,height=720,framerate=30/1" ! capsfilter name=n1_caps_1 caps="video/x-raw,width=1280,height=720,framerate=30/1" ! videoconvert name=n2_videoconvert_1 ! capsfilter name=n3_caps_1 caps="video/x-raw,format=NV12,width=1280,height=720,framerate=30/1" ! neatencoder name=n4_encoder_1 enc-type=h264 enc-profile=baseline enc-level=4.0 enc-fmt=NV12 enc-width=1280 enc-height=720 enc-frame-rate=30 enc-bitrate=4000 enc-ip-mode=async ip-rate-ctrl=false ! h264parse name=n5_h264parse_1 disable-passthrough=true config-interval=1 ! rtph264pay name=pay0 pt=96 config-interval=1 ! udpsink host=192.168.131.12 port=9000 sync=false async=false


## Seed And Build

The seed frame is a mid-gray NV12 buffer of exactly the declared size. It is thrown away; its only job
is to give caps negotiation something concrete to agree on.

In [13]:
seed_nv12 = np.full((FRAME_HEIGHT * 3 // 2, FRAME_WIDTH), 128, dtype=np.uint8)  # UV = 128 is neutral gray.
seed_nv12[:FRAME_HEIGHT, :] = 16  # Y = 16 is black in limited-range video.
seed_tensor = make_nv12_tensor(seed_nv12, FRAME_WIDTH, FRAME_HEIGHT)

video_run = video_graph.build([seed_tensor])
print("video sender running, pushing to", f"{sender_opt.host}:{sender_opt.video_port}")

[1/4] Initializing runtime...


video sender running, pushing to 192.168.131.12:9000


[2/4] Preparing input stream...
[3/4] Building graph...
[4/4] Graph ready (103 ms)


## Push Frames

`video_run.push([tensor])` hands one frame to the encoder. It returns `False` when the graph refused
the frame, which almost always means the tensor did not match the declared contract.

The cell below generates a synthetic test pattern: a moving color bar and a frame counter. You do not
need a camera or a model to see the video sender work. Open the Insight viewer on channel `CHANNEL`
while this runs and you will see the bar move.

In [14]:
def make_test_frame(index: int, width: int, height: int):
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    # Static color bars so a frozen stream is obvious at a glance.
    bar_w = width // 6
    colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (0, 255, 255), (255, 0, 255), (255, 255, 0)]
    for i, color in enumerate(colors):
        frame[:, i * bar_w:(i + 1) * bar_w] = color
    # A sweeping white bar makes motion, and therefore inter-frame encoding, visible.
    x = int((index * 12) % max(1, width - 60))
    frame[:, x:x + 60] = (255, 255, 255)
    cv2.putText(frame, f"neat video_sender  frame {index}", (40, height - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3, cv2.LINE_AA)
    return frame


FRAMES_TO_SEND = 300  # About 10 seconds at 30 fps.
frame_interval = 1.0 / max(1, FRAME_FPS)

sent = 0
for index in range(FRAMES_TO_SEND):
    started = time.perf_counter()
    nv12 = bgr_to_nv12(make_test_frame(index, FRAME_WIDTH, FRAME_HEIGHT))
    if not video_run.push([make_nv12_tensor(nv12, FRAME_WIDTH, FRAME_HEIGHT)]):
        raise RuntimeError("video push failed: tensor did not match the declared contract")
    sent += 1
    # Pace the loop, because sync=False means the sender will not pace it for us.
    time.sleep(max(0.0, frame_interval - (time.perf_counter() - started)))

print("frames pushed:", sent)

frames pushed: 300


## Confirm The Bytes Really Left The Board

A successful `push()` only means the graph accepted the frame. It does not prove that RTP reached the
receiver. Insight exposes an ingest statistics endpoint that answers that question directly, without
involving a browser at all.

A healthy channel shows `rtp.packets_received` climbing, a nonzero `rtp.bitrate_bps`, and
`media.seen_sps` / `media.seen_pps` true. Those last two are the `config_interval` setting paying off:
they are the stream headers, and a viewer cannot decode without them.

In [15]:
def insight_get(path: str):
    # Insight uses a local mkcert certificate, so skip verification for this diagnostic call.
    import ssl

    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    with urllib.request.urlopen(f"{INSIGHT_API}{path}", context=ctx, timeout=5) as response:
        return json.loads(response.read().decode("utf-8"))


try:
    stats = insight_get("/api/ingest/stats")
    for channel in stats.get("channels", []):
        if channel.get("channel") != CHANNEL:
            continue
        rtp = channel.get("rtp", {})
        media = channel.get("media", {})
        print("channel :", channel.get("channel"))
        print("packets :", rtp.get("packets_received"))
        print("bitrate :", rtp.get("bitrate_bps"), "bps")
        print("seen_sps:", media.get("seen_sps"), " seen_pps:", media.get("seen_pps"))
        print("idr_count:", media.get("idr_count"))
        break
    else:
        print("channel", CHANNEL, "is not active. Raw payload:")
        print(json.dumps(stats, indent=2)[:800])
except Exception as exc:
    print("Insight not reachable, skipping verification:", type(exc).__name__, exc)
    print("The push loop above still succeeded; this cell only checks the receiver side.")

channel : 0
packets : 4144
bitrate : 302431.8 bps
seen_sps: True  seen_pps: True
idr_count: 22


## Pattern: Encoded Pass-Through

When you already have H.264 and only want to move it onto RTP/UDP, do not decode and re-encode. Use the
encoded input kind. The group then builds only parser, packetizer, and UDP sink, and the frames never
touch the hardware encoder.

This is the right choice for a recorder or a relay. It is the wrong choice the moment your app needs to
look at or draw on the pixels, because you cannot annotate bytes you never decoded.

In [16]:
passthrough_opt = pyneat.VideoSenderOptions.h264_rtp_udp_from_encoded()
passthrough_opt.host = INSIGHT_HOST
passthrough_opt.channel = 1  # A different viewer tile from the raw sender above.
passthrough_opt.video_port_base = VIDEO_PORT_BASE
passthrough_opt.rtp.payload_type = 96
passthrough_opt.rtp.config_interval = 1

print("encoded pass-through")
print("  encodes?    :", passthrough_opt.is_raw_input())
print("  destination :", f"{passthrough_opt.host}:{passthrough_opt.video_port}")
print("  encoder settings are present but unused:", passthrough_opt.encoder.bitrate_kbps)

encoded pass-through
  encodes?    : False
  destination : 192.168.131.12:9001
  encoder settings are present but unused: 4000


To build a real pass-through graph, pair it with `pyneat.groups.rtsp_encoded_input(...)`, which pulls
H.264 off an RTSP source without decoding it. Notebook 07 shows the decoded path end to end; the
encoded path is the same shape with the decoder removed:

```python
graph = pyneat.Graph("relay")
graph.connect(
    pyneat.groups.rtsp_encoded_input(encoded_source_options),
    pyneat.groups.video_sender(passthrough_opt),
)
```

## Pattern: Multi-Stream Fan-Out

Each stream gets its own channel, and therefore its own port and its own viewer tile. Nothing else in
the options needs to change. Insight accepts channels `0` through `79`.

In [17]:
def build_stream_sender(channel: int, width: int, height: int, fps: int):
    opt = make_sender_options(channel, width, height, fps)
    graph = pyneat.Graph(f"video_sender_ch{channel}")
    graph.add(pyneat.nodes.input(make_nv12_input_options(width, height, fps)))
    graph.add(pyneat.groups.video_sender(opt))
    return graph, opt


for channel in range(4):
    _graph, opt = build_stream_sender(channel, 640, 480, 30)
    print(f"stream {channel}: {opt.host}:{opt.video_port}  tile src={channel}")

stream 0: 192.168.131.12:9000  tile src=0
stream 1: 192.168.131.12:9001  tile src=1
stream 2: 192.168.131.12:9002  tile src=2
stream 3: 192.168.131.12:9003  tile src=3


## Stop The Run

A `Run` holds the encoder and an open UDP socket. Stop it when you are done, or the next build in the
same kernel may fight it for the port.

In [18]:
video_run.stop()
print("video sender stopped")

video sender stopped


## Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `push()` returns `False` | Tensor does not match the declared contract. | Check width, height, and `PixelFormat.NV12`; confirm the tensor has Y and UV planes. |
| `RuntimeError` on `build()` | `InputOptions` caps disagree with `h264_rtp_udp_from_raw` args. | Make one function own width/height/fps and feed both from it. |
| Viewer tile stays black, packets are arriving | No SPS/PPS, so the browser cannot start a decode. | Set `rtp.config_interval = 1`. |
| Video is blocky or smeared | Bitrate too low for the scene. | Raise `encoder.bitrate_kbps`. |
| Encoder rejects the stream | `level` too low for resolution x frame rate. | Raise `encoder.level`, e.g. `"4.2"`. |
| Nothing at all reaches the host | Wrong `host`, or a firewall between DevKit and Insight. | Run the loopback cell above. If loopback works, the sender is fine. |
| Stream lags further behind over time | `sync = True` on a live source. | Set `sync = False` and `async_ = False`. |
| Colors look wrong | BGR bytes pushed as NV12. | Convert with `bgr_to_nv12` before building the tensor. |

The single most useful diagnostic is the loopback cell. It cleanly separates "my sender is broken" from
"my network or receiver is broken", and those two problems have nothing in common.

## What To Remember

- `video_sender` is a graph fragment, not a node. It expands into caps, convert, encode, parse,
  packetize, and UDP send.
- Pick the input kind with a factory: `h264_rtp_udp_from_raw(w, h, fps)` to encode, or
  `h264_rtp_udp_from_encoded()` to remux existing H.264.
- `video_port` is derived: `video_port_base + channel`. Change the channel, never the port.
- The `InputOptions` caps, the factory arguments, and the tensors you push must agree exactly.
- NV12 needs a real tensor with `ImageSpec` and Y/UV planes, not a bare numpy array.
- `rtp.config_interval = 1` is what lets a viewer join a stream that is already running.
- Keep `sync` and `async_` off for live analytics.
- A successful `push()` proves the graph accepted the frame, not that the network delivered it.
  Confirm with `/api/ingest/stats` or a UDP loopback probe.
- Video and metadata for the same stream must share a channel number. Notebook 06 covers metadata.

## References

- Notebook 02, `02_rtsp_input_and_decode_options.ipynb`, for the ingest side.
- Notebook 06, `06_metadata_sender_options.ipynb`, for the JSON metadata side.
- Notebook 07, `07_rtsp_decode_encode_metadata_to_insight.ipynb`, for the full RTSP-to-Insight pipeline.
- Core header: `include/nodes/groups/VideoSender.h`.
- Demo app: `apps/single-stream-yolo-yolo11/main.py`, function `build_video_graph`.